# PySpark reference 13: performance and optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-13-performance.ipynb)

Diagnosing with the Spark UI and the noop sink, sizing shuffle partitions and AQE, avoiding and reusing shuffles, column, predicate and dynamic partition pruning, small files, detecting and fixing skew, caching, the memory model, code-level anti-patterns, a configuration cheat sheet and a tuning checklist. Practise it in [PySpark practice 13: performance and optimization](pyspark-practice-13-performance.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. `count_ops(df, name)` counts an operator in a plan and `timed_noop(df)` runs a query without output and returns its duration.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import glob
import os
import time as _time


def count_ops(df, name):
    """How many times an operator name appears in df's physical plan."""
    return sum(name in line for line in plan_lines(df))


def shuffles(df):
    """Shuffle exchanges in df's physical plan (BroadcastExchange is not a shuffle)."""
    return sum("Exchange " in line and "BroadcastExchange" not in line for line in plan_lines(df))


def timed_noop(df):
    """Run df fully without writing output (the noop sink) and return seconds taken."""
    start = _time.time()
    df.write.format("noop").mode("overwrite").save()
    return round(_time.time() - start, 2)

## Contents

| section | entries |
| --- | --- |
| [Diagnosing](#diagnosing) | [What to look at in the Spark UI](#what-to-look-at-in-the-spark-ui), [Benchmarking a query with the `noop` sink](#benchmarking-a-query-with-the-noop-sink) |
| [Shuffles and partitions](#shuffles-and-partitions) | [Sizing shuffle partitions and AQE coalescing](#sizing-shuffle-partitions-and-aqe-coalescing), [Avoiding shuffles](#avoiding-shuffles), [Exchange reuse](#exchange-reuse) |
| [Reading less data](#reading-less-data) | [Column pruning, predicate pushdown and partition pruning](#column-pruning-predicate-pushdown-and-partition-pruning), [Dynamic partition pruning](#dynamic-partition-pruning), [Small files](#small-files) |
| [Skew](#skew) | [Detecting skew](#detecting-skew), [Fixing skew](#fixing-skew) |
| [Caching and memory](#caching-and-memory) | [When caching helps and when it hurts](#when-caching-helps-and-when-it-hurts), [The executor memory model and spills](#the-executor-memory-model-and-spills) |
| [Code-level anti-patterns](#code-level-anti-patterns) | [Patterns that silently cost time](#patterns-that-silently-cost-time) |
| [Configuration cheat sheet](#configuration-cheat-sheet) | [Settings worth knowing](#settings-worth-knowing) |
| [Tuning checklist](#tuning-checklist) | [A repeatable process](#a-repeatable-process) |

## Diagnosing

### What to look at in the Spark UI

Start every tuning task from evidence. The Spark UI (port 4040 for a running application, or the history server)
shows where time goes:

| tab | look for | it suggests |
| --- | --- | --- |
| Jobs / Stages | the stage that takes most of the time | where to focus |
| Stage → task metrics | max task time far above the median | **skew** |
| Stage → shuffle read / write | gigabytes moved per stage | an avoidable shuffle, or a join to broadcast |
| Stage → spill (memory / disk) | non-zero spill | partitions too large for executor memory |
| Stage → GC time | GC a large share of task time | memory pressure, too-large partitions, caching too much |
| SQL tab → plan with metrics | rows output per operator, `number of files read`, `pruned partitions` | missing pruning, exploding joins |
| Executors | failed tasks, lost executors, uneven input | OOM, bad nodes, skew |

**Notes:** change one thing at a time and measure; confirm the output is unchanged after each change.

### Benchmarking a query with the `noop` sink

`df.write.format("noop").mode("overwrite").save()` executes the whole query, including every shuffle, but writes nothing.
It measures compute time without the cost of `collect()` to the driver or of writing files, and is the fair way to
compare two versions of a query.

**Notes:** run each variant at least twice (the first run also pays JIT and cache warm-up), and compare on realistic data
sizes: tiny local data hides most effects.

In [ ]:
facts = spark.range(2_000_000).select((F.col("id") % 50_000).alias("customer_id"), (F.rand(seed=1) * 100).alias("amount"))
dim = spark.range(50_000).select(F.col("id").alias("customer_id"), (F.col("id") % 5).cast("string").alias("segment"))
q = facts.join(dim, "customer_id").groupBy("segment").agg(F.sum("amount"))
timed_noop(q)  # warm-up
print("seconds:", timed_noop(q))

## Shuffles and partitions

### Sizing shuffle partitions and AQE coalescing

Every shuffle writes `spark.sql.shuffle.partitions` partitions (default 200). Too few: huge partitions that spill and
straggle. Too many: thousands of tiny tasks and files. AQE **coalesces** small post-shuffle partitions toward
`spark.sql.adaptive.advisoryPartitionSizeInBytes` (64 MB), so a generous `shuffle.partitions` plus AQE is a good
default.

| setting | default | meaning |
| --- | --- | --- |
| `spark.sql.shuffle.partitions` | 200 | partitions after a shuffle, before AQE |
| `spark.sql.adaptive.coalescePartitions.enabled` | true | merge small partitions |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | 64MB | target size when coalescing (and splitting skew) |
| `spark.sql.adaptive.coalescePartitions.minPartitionSize` | 1MB | lower bound for coalesced partitions |

**Rule of thumb:** aim for post-shuffle partitions of 100–200 MB and at least 2–3 tasks per executor core.

**Notes:** partitions smaller than `minPartitionSize` are merged regardless of the advisory size, so on small data AQE
often coalesces everything into one partition.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "64")
variants = [
    ("defaults", {}),
    ("advisory 256KB, min 16KB", {"spark.sql.adaptive.advisoryPartitionSizeInBytes": "256KB", "spark.sql.adaptive.coalescePartitions.minPartitionSize": "16KB"}),
    ("coalescing disabled", {"spark.sql.adaptive.coalescePartitions.enabled": "false"}),
]
for label, conf in variants:
    for k, v in conf.items():
        spark.conf.set(k, v)
    done = facts.groupBy("customer_id").agg(F.sum("amount").alias("s"))
    done.collect()
    print(f"{label:26} {done.rdd.getNumPartitions():3} partitions (from 64)")
    for k in conf:
        spark.conf.unset(k)
spark.conf.set("spark.sql.shuffle.partitions", "4")

### Avoiding shuffles

| technique | why it helps |
| --- | --- |
| filter and select columns before joins and aggregations | less data per shuffle; Spark pushes many filters down itself |
| broadcast small sides of joins | no shuffle of the large side ([joins](pyspark-reference-09-joins.ipynb#broadcast-hash-join)) |
| aggregate before joining when the result is smaller | joins fewer rows |
| reuse one partitioning for several operations on the same key | Spark skips an `Exchange` when the data is already partitioned that way |
| avoid `orderBy` unless the output must be sorted | a global sort is a range shuffle plus sampling |
| `dropDuplicates(subset)` / `distinct` only where needed | each is a full shuffle |

**Notes:** count shuffle `Exchange` operators in the plan before and after a change (`BroadcastExchange` is not a
shuffle). The `shuffles(df)` helper does exactly that.

In [ ]:
per_cust = facts.groupBy("customer_id").agg(F.sum("amount").alias("s"))
same_key = per_cust.withColumn("r", F.rank().over(Window.partitionBy("customer_id").orderBy("s")))
other_key = per_cust.withColumn("r", F.rank().over(Window.partitionBy(F.col("customer_id") % 7).orderBy("s")))
print("groupBy + window on the same key:", shuffles(same_key), "shuffle(s) | on a different key:", shuffles(other_key))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("join + groupBy, sort-merge:", shuffles(facts.join(dim, "customer_id").groupBy("segment").count()),
      "| with broadcast of dim:", shuffles(facts.join(F.broadcast(dim), "customer_id").groupBy("segment").count()))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

### Exchange reuse

When the same shuffle appears twice in one plan (a DataFrame used in two branches of a union or self join), Spark
computes it once and reads it again: the final plan (after the query runs) shows `ReusedExchange`. This only works
**within one query**; across separate actions, cache the DataFrame instead.

In [ ]:
per_customer = facts.groupBy("customer_id").agg(F.sum("amount").alias("s"))
both = per_customer.where("s > 5000").unionByName(per_customer.where("s < 100"))
print("ReusedExchange in the final plan:", "ReusedExchange" in final_plan(both))

## Reading less data

### Column pruning, predicate pushdown and partition pruning

| optimization | how to see it | what blocks it |
| --- | --- | --- |
| column pruning | `ReadSchema` lists only needed columns | `select("*")` carried to the end, UDFs over whole rows |
| predicate pushdown | `PushedFilters` in the scan | casts on the column, UDFs, functions the source cannot evaluate |
| partition pruning | `PartitionFilters`; fewer files read | filtering on an expression of the partition column instead of the column itself |
| dynamic partition pruning (DPP) | `dynamicpruningexpression` in the fact scan | join keys that are not partition columns; disabled setting |

**Notes:** Parquet row-group statistics let pushed filters skip data inside files; sorting data by a commonly filtered
column before writing (`sortWithinPartitions`) makes those statistics much more selective.

In [ ]:
pq = f"{WORK_DIR}/orders_pq"
orders.write.mode("overwrite").parquet(pq)
scan = spark.read.parquet(pq)

def scan_info(df):
    return [l.strip()[:85] for l in plan_lines(df, "formatted") if l.strip().startswith(("PushedFilters", "ReadSchema"))]

print("good:          ", scan_info(scan.where(F.col("amount") > 100).select("order_id")))
print("cast blocks it:", scan_info(scan.where(F.col("amount").cast("string") == "120.00").select("order_id")))
not_pushable = F.udf(lambda a: float(a) > 100, "boolean")
print("udf blocks it: ", scan_info(scan.where(not_pushable("amount")).select("order_id")))

### Dynamic partition pruning

When a large table partitioned by a key is joined to a filtered small table on that key, Spark runs the small side
first and uses its keys to skip partitions of the large table **at run time**. The scan of the fact table shows
`dynamicpruningexpression(...)` in its partition filters.

| setting | default |
| --- | --- |
| `spark.sql.optimizer.dynamicPartitionPruning.enabled` | true |

In [ ]:
fact_path = f"{WORK_DIR}/fact_by_customer"
orders.write.mode("overwrite").partitionBy("customer_id").parquet(fact_path)
fact = spark.read.parquet(fact_path)
north = customers.where("region = 'North'")
pruned = fact.join(north, "customer_id")
print([l.strip()[:110] for l in plan_lines(pruned, "formatted") if "dynamicpruning" in l][:1])
print("rows:", pruned.count())

### Small files

Thousands of small files make every read slow: one task (and one metadata call) per file, poor compression and no
useful statistics. They come from writing many partitions with many tasks, streaming micro-batches, and frequent appends.

| fix | how |
| --- | --- |
| fewer, larger output files | `repartition(n)` or `repartition(partition_col)` before `partitionBy` writes |
| cap file size | `maxRecordsPerFile` |
| compact afterwards | rewrite with fewer files; Delta `OPTIMIZE` |
| read side | `spark.sql.files.maxPartitionBytes` and `openCostInBytes` pack small files into larger read partitions |

In [ ]:
messy = f"{WORK_DIR}/messy"
orders.repartition(12).write.mode("overwrite").partitionBy("status").parquet(messy)
print("files written with 12 tasks:", len(glob.glob(f"{messy}/*/*.parquet")))
orders.repartition("status").write.mode("overwrite").partitionBy("status").parquet(messy)
print("files after repartition by the partition column:", len(glob.glob(f"{messy}/*/*.parquet")))

## Skew

### Detecting skew

Skew means a few keys hold most of the rows, so the tasks that get those keys run far longer than the rest. Signs:
one task much slower than the median in a stage, spill on a few tasks only, and executors idle while one finishes.
Measure the key distribution directly before and after a fix.

In [ ]:
skewed = spark.range(100_000).select(F.when(F.col("id") < 80_000, 0).otherwise(F.col("id") % 1000).alias("key"), F.lit(1).alias("v"))
dist = skewed.groupBy("key").count()
total = skewed.count()
dist.orderBy(F.desc("count")).limit(3).withColumn("share", F.round(F.col("count") / total, 3)).show()
(skewed.repartition(8, "key").groupBy(F.spark_partition_id().alias("partition")).count().orderBy(F.desc("count")).show(3))

### Fixing skew

| technique | where | how |
| --- | --- | --- |
| AQE skew join | sort-merge joins | on by default; splits partitions above `skewedPartitionFactor` × median and `skewedPartitionThresholdInBytes` |
| broadcast the other side | joins with a small side | no shuffle, so no skewed partition |
| filter or handle junk keys | `null`, `-1`, `unknown` keys | drop them or process them separately |
| salting | joins and aggregations | spread a hot key over N sub-keys, then combine ([joins](pyspark-reference-09-joins.ipynb#salting-a-hot-key)) |
| two-stage aggregation | `groupBy` on a hot key | aggregate by `(key, salt)` first, then by `key` |
| isolate hot keys | joins | broadcast-join the few hot keys, sort-merge-join the rest, `union` the results |

In [ ]:
N = 8
salted = (skewed.withColumn("salt", (F.rand(seed=3) * N).cast("int"))
          .groupBy("key", "salt").agg(F.sum("v").alias("partial"))
          .groupBy("key").agg(F.sum("partial").alias("total")))
plain = skewed.groupBy("key").agg(F.sum("v").alias("total"))
print("same result:", salted.exceptAll(plain).count() == 0 and plain.exceptAll(salted).count() == 0)
print("largest stage-one group, plain vs salted:", skewed.groupBy("key").count().agg(F.max("count")).first()[0],
      skewed.withColumn("salt", (F.rand(seed=3) * N).cast("int")).groupBy("key", "salt").count().agg(F.max("count")).first()[0])

## Caching and memory

### When caching helps and when it hurts

Cache a DataFrame only when it is **expensive to compute** and **used by more than one action**. Otherwise caching adds a
full materialization and takes executor memory away from shuffles and joins.

| cache | do not cache |
| --- | --- |
| a cleaned table feeding several outputs | a DataFrame used once |
| an iterative algorithm reusing the same input | the raw input of a single linear pipeline |
| a small lookup reused in many joins (or broadcast it) | data larger than cluster memory without `MEMORY_AND_DISK` |

**Notes:** materialize with an action (`count()`), check the Storage tab, and `unpersist()` when done. In one query, Spark
already reuses repeated subplans (`ReusedExchange`) without caching.

### The executor memory model and spills

An executor's heap is split into **reserved** memory (300 MB), **user** memory (your Python and UDF objects on the JVM side,
metadata), and **unified** memory (`spark.memory.fraction`, 60% of the rest) shared by **execution** (shuffles, joins,
sorts, aggregations) and **storage** (cache, broadcasts). When execution needs more than it has, it **spills** to disk;
when even that is not enough, tasks fail with out-of-memory errors.

| setting | purpose |
| --- | --- |
| `spark.executor.memory` | JVM heap per executor |
| `spark.executor.memoryOverhead` | off-heap for Python workers, Arrow, native libraries (raise it for pandas UDFs) |
| `spark.memory.fraction` / `spark.memory.storageFraction` | unified memory share / storage share protected from eviction |
| `spark.executor.cores` | concurrent tasks per executor (memory per task = unified memory / cores) |
| `spark.driver.memory`, `spark.driver.maxResultSize` | driver heap, and the cap on results collected to the driver |

**Common causes of OOM:** `collect()` / `toPandas()` of big data (driver), broadcasting a table that is not small, one
skewed partition or group, `collect_list` over huge groups, pandas UDFs with large groups (overhead memory), too many cores
per executor for the partition size.

## Code-level anti-patterns

### Patterns that silently cost time

| anti-pattern | better |
| --- | --- |
| `withColumn` in a Python loop over many columns | one `select` or `withColumns` (one projection instead of hundreds) |
| `df.count() == 0` / `len(df.collect()) == 0` to test emptiness | `df.isEmpty()` (stops at the first row) |
| repeated actions on an uncached, expensive DataFrame | cache it, or restructure into one action |
| Python UDFs for string, date or array logic | built-in functions and higher-order functions |
| `collect()` then loop in Python | keep the logic in Spark; collect only small results |
| `orderBy` before writing when order is not needed | `sortWithinPartitions` or nothing |
| `explode` of large arrays before filtering | filter the array with `filter()` first |
| UDFs or casts in join keys and filters | precompute and type columns once |

In [ ]:
wide = spark.range(10).select("id")
looped = wide
for i in range(50):
    looped = looped.withColumn(f"c{i}", F.col("id") + i)
single = wide.select("id", *[(F.col("id") + i).alias(f"c{i}") for i in range(50)])
def analyzed_projects(df):
    text = "\n".join(plan_lines(df, "extended"))
    return text.split("== Analyzed Logical Plan ==")[1].split("== Optimized Logical Plan ==")[0].count("Project [")
print("analyzed Project nodes, loop vs select:", analyzed_projects(looped), analyzed_projects(single))
print("isEmpty on a filtered DataFrame:", facts.where("amount > 1000").isEmpty())

## Configuration cheat sheet

### Settings worth knowing

| setting | default | when to change |
| --- | --- | --- |
| `spark.sql.shuffle.partitions` | 200 | raise for very large shuffles; AQE coalesces small ones |
| `spark.sql.adaptive.enabled` | true | keep on |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | 64MB | larger for big clusters and files |
| `spark.sql.autoBroadcastJoinThreshold` | 10MB | raise (e.g. 100MB) if dimensions fit in executor memory; `-1` to disable |
| `spark.sql.adaptive.skewJoin.enabled` | true | keep on |
| `spark.sql.files.maxPartitionBytes` | 128MB | read partition size |
| `spark.sql.files.openCostInBytes` | 4MB | higher packs more small files per read task |
| `spark.sql.optimizer.dynamicPartitionPruning.enabled` | true | keep on |
| `spark.sql.execution.arrow.pyspark.enabled` | false | true for `toPandas` / `createDataFrame(pandas)` |
| `spark.sql.parquet.compression.codec` | snappy | `zstd` for better compression |
| `spark.serializer` | Java | `org.apache.spark.serializer.KryoSerializer` for RDD-heavy code |
| `spark.dynamicAllocation.enabled` | false | true on shared clusters to scale executors with load |

In [ ]:
for key in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.sql.autoBroadcastJoinThreshold",
            "spark.sql.files.maxPartitionBytes", "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    print(f"{key:52} {spark.conf.get(key)}")

## Tuning checklist

### A repeatable process

1. **Measure** the baseline on realistic data (`noop` sink, Spark UI); keep the output for comparison.
2. **Find the slow stage** and classify it: too much data read, too much shuffled, skew, spill, or Python overhead.
3. **Read less:** prune columns, push filters, partition and sort data by common filters, compact small files.
4. **Shuffle less:** broadcast small sides, aggregate early, reuse partitioning, drop unneeded `distinct`/`orderBy`.
5. **Balance work:** size shuffle partitions, let AQE coalesce and split skew, salt hot keys.
6. **Fit in memory:** smaller partitions, fewer cores per executor, more overhead for Python, avoid driver collects.
7. **Remove Python from the hot path:** built-ins and SQL UDFs over Python UDFs; pandas UDFs when Python is required.
8. **Verify** the result is identical (`exceptAll` both ways) and record the gain.

**Next:** practise these entries in [PySpark practice 13: performance and optimization](pyspark-practice-13-performance.ipynb).